In [ ]:
import os
import requests
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# ==========================================
# CONFIGURAÇÕES
# ==========================================

INPUT_IMAGE_URL = "https://cdn.myanimelist.net/images/anime/1015/138006.jpg" 
IMAGE_SIZE = (200, 200) 
N_CANAIS = 3  

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

# Identifica se a nuvem possui GPU disponível
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

# ==========================================
# FUNÇÃO AUXILIAR DE DOWNLOAD E PROCESSAMENTO
# ==========================================

def baixar_e_processar_imagem(url, target_size):
    try:
        response = requests.get(url, headers=HEADERS, timeout=5)
        response.raise_for_status()
        img_arr = np.array(bytearray(response.content), dtype=np.uint8)
        img = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Redimensiona para o tamanho alvo (200x200) usando interpolação de área de alta qualidade
        img_redimensionada = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)
        return img_redimensionada.astype(np.float32) / 255.0
    except Exception as e:
        return None

# ==========================================
# ARQUITETURA DA CNN (ScoreCNN)
# ==========================================

class ScoreCNN(nn.Module):
    def __init__(self, n_canais=3):
        super().__init__()
        entrada_canais = n_canais + 1  # 3 canais RGB + 1 canal para o nível de sigma
        self.net = nn.Sequential(
            nn.Conv2d(entrada_canais, 32, kernel_size=3, padding=1),
            nn.GroupNorm(8, 32),
            nn.SiLU(),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.GroupNorm(8, 64),
            nn.SiLU(),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.GroupNorm(8, 64),
            nn.SiLU(),

            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.GroupNorm(8, 32),
            nn.SiLU(),

            nn.Conv2d(32, n_canais, kernel_size=3, padding=1),  
        )

    def forward(self, x):
        return self.net(x)

# ==========================================
# DATASET VIRTUAL DINÂMICO (SOLUÇÃO ANTI-CRASH RAM)
# ==========================================

class ScoreDataset(Dataset):
    """Dataset que recebe imagens LIMPAS e gera as perturbações de ruído on-the-fly."""
    def __init__(self, imgs_limpas, sigmas, n_perturbacoes, img_shape):
        self.imgs_limpas = imgs_limpas  # Armazena apenas as imagens base originais
        self.sigmas = sigmas            
        self.n_perturbacoes = n_perturbacoes
        self.H, self.W, self.C = img_shape
        
        # Define o tamanho combinatório total virtual
        self.total_amostras = len(imgs_limpas) * len(sigmas) * n_perturbacoes

    def __len__(self):
        return self.total_amostras

    def __getitem__(self, idx):
        # Mapeamento aritmético do índice virtual para a combinação real correspondente
        idx_img = idx // (len(self.sigmas) * self.n_perturbacoes)
        resto = idx % (len(self.sigmas) * self.n_perturbacoes)
        idx_sigma = resto // self.n_perturbacoes
        
        # 1. Recupera e reconverte a imagem limpa para 3D (H, W, C) e depois para (C, H, W) do PyTorch
        x_0 = self.imgs_limpas[idx_img].reshape(self.H, self.W, self.C).transpose(2, 0, 1)
        sigma = self.sigmas[idx_sigma]
        
        # 2. Gera o ruído gaussiano puro (Z) para esta chamada
        z = np.random.normal(0, 1, x_0.shape).astype(np.float32)
        
        # 3. Aplica a equação de perturbação
        x_ruidoso = x_0 + sigma * z
        
        # 4. Injeta o canal extra com o valor escalar de sigma constante (1, H, W)
        sigma_channel = np.full((1, self.H, self.W), sigma, dtype=np.float32)
        x_entrada = np.concatenate([x_ruidoso, sigma_channel], axis=0)
        
        # O alvo do modelo de Score Matching é o ruído invertido (-z)
        score_alvo = -z
        
        return torch.from_numpy(x_entrada).float(), torch.from_numpy(score_alvo).float()

# ==========================================
# INFERÊNCIA / PREDIÇÃO DO SCORE
# ==========================================

@torch.no_grad()
def prever_score_cnn(modelo, x_vetor_ruidoso, sigma, img_shape):
    H, W, C = img_shape
    modelo.eval()

    img = x_vetor_ruidoso.reshape(H, W, C).astype(np.float32).transpose(2, 0, 1)  
    sigma_channel = np.full((1, H, W), sigma, dtype=np.float32)
    x = np.concatenate([img, sigma_channel], axis=0)[None, ...]  

    x_tensor = torch.from_numpy(x).to(device)
    
    # Se estiver usando GPU, aproveita inferência acelerada em float16
    if device.type == 'cuda':
        with torch.amp.autocast('cuda'):
            pred = modelo(x_tensor)
    else:
        pred = modelo(x_tensor)
        
    pred = pred.squeeze(0).cpu().numpy().transpose(1, 2, 0)  
    return pred.flatten() / sigma  

# ==========================================
# FLUXO PRINCIPAL EXECUTÁVEL
# ==========================================

if __name__ == "__main__":
    img_shape = (IMAGE_SIZE[0], IMAGE_SIZE[1], N_CANAIS)  # (200, 200, 3)

    print("--- PASSO 1: Carregando banco de dados de Animes ---")
    if not os.path.exists("banco_de_dados_animes_limpo.csv"):
        raise FileNotFoundError("Por favor, certifique-se de que o arquivo 'banco_de_dados_animes_limpo.csv' está na mesma pasta.")
        
    df = pd.read_csv("banco_de_dados_animes_limpo.csv")
    urls_treino = df['Image_URL'].dropna().tolist()
    
    print("Baixando e redimensionando imagens para criar o conjunto de dados base...")
    imagens_treino = []
    LIMITE_IMAGENS = 1050
    
    barra_download = tqdm(urls_treino, total=min(LIMITE_IMAGENS, len(urls_treino)), desc="Baixando imagens", unit="img")
    for url in barra_download:
        img = baixar_e_processar_imagem(url, IMAGE_SIZE)
        if img is not None:
            imagens_treino.append(img.flatten()) 
        if len(imagens_treino) >= LIMITE_IMAGENS:
            break
    barra_download.close()
            
    if len(imagens_treino) == 0:
        print("Falha ao baixar imagens. Utilizando dados sintéticos estruturados...")
        imagens_treino = [np.random.rand(IMAGE_SIZE[0] * IMAGE_SIZE[1] * N_CANAIS) for _ in range(5)]
        
    X_dados = np.array(imagens_treino)
    print(f"Dataset base carregado na memória. Formato: {X_dados.shape}")

    print("\n--- PASSO 2: Configurando DataLoader Otimizado para Nuvem ---")
    sigmas = np.array([1.0, 0.5, 0.2, 0.1, 0.05])  
    n_perturbacoes = 80  
    
    # Criando o Dataset dinâmico virtual
    dataset = ScoreDataset(X_dados, sigmas, n_perturbacoes, img_shape)
    
    # DataLoader configurado para instâncias na Nuvem (com paralelismo de CPU e pinning de memória para GPU)
    num_workers_nuvem = 4 if device.type == 'cuda' else 0
    loader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=num_workers_nuvem, pin_memory=True)

    print(f"Dataset virtual configurado com {len(dataset)} amostras geradas dinamicamente.")
    print("Uso de memória RAM estabilizado de forma segura!")

    modelo_cnn = ScoreCNN(n_canais=N_CANAIS).to(device)
    otimizador = optim.Adam(modelo_cnn.parameters(), lr=1e-3)
    criterio = nn.MSELoss()

    # Ativador de Precisão Mista para GPUs modernas na Nuvem
    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

    print("\nIniciando Treinamento da ScoreCNN...")
    EPOCHS = 10  # Ajuste as épocas conforme necessário para convergência
    modelo_cnn.train()

    barra_epocas = tqdm(range(1, EPOCHS + 1), desc="Treinando CNN", unit="época")
    for epoch in barra_epocas:
        perda_total = 0.0

        barra_batches = tqdm(loader, desc=f"  Época {epoch}/{EPOCHS}", unit="batch", leave=False)
        for x_batch, y_batch in barra_batches:
            x_batch = x_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)

            otimizador.zero_grad()
            
            # Treinamento acelerado por hardware via AMP se CUDA estiver ativo
            if device.type == 'cuda':
                with torch.amp.autocast('cuda'):
                    pred = modelo_cnn(x_batch)
                    perda = criterio(pred, y_batch)
                scaler.scale(perda).backward()
                scaler.step(otimizador)
                scaler.update()
            else:
                pred = modelo_cnn(x_batch)
                perda = criterio(pred, y_batch)
                perda.backward()
                otimizador.step()

            perda_total += perda.item() * x_batch.size(0)
            barra_batches.set_postfix(mse=f"{perda.item():.4f}")

        perda_media = perda_total / len(dataset)
        barra_epocas.set_postfix(mse_medio=f"{perda_media:.6f}")

    print("Modelo treinado com sucesso!")


Usando dispositivo: cpu
--- PASSO 1: Carregando banco de dados de Animes ---
Baixando e redimensionando imagens para criar o conjunto de dados base...


Baixando imagens: 100%|██████████| 1050/1050 [03:19<00:00,  5.27img/s]


Dataset base carregado na memória. Formato: (1048, 120000)

--- PASSO 2: Configurando DataLoader Otimizado para Nuvem ---
Dataset virtual configurado com 419200 amostras geradas dinamicamente.
Uso de memória RAM estabilizado de forma segura!

Iniciando Treinamento da ScoreCNN...


Treinando CNN:   0%|          | 0/10 [00:00<?, ?época/s]/Users/juliano/Projeto1/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Treinando CNN:   0%|          | 0/10 [00:00<?, ?época/s]


RuntimeError: Input type (double) and bias type (float) should be the same

In [ ]:

print("\n--- PASSO 3: Baixando Imagem de Entrada do Usuário ---")
img_alvo = baixar_e_processar_imagem(INPUT_IMAGE_URL, IMAGE_SIZE)
if img_alvo is None:
    print("Não foi possível acessar a URL informada. Gerando ruído randômico de base.")
    img_alvo = np.random.uniform(0, 1, img_shape)
        
vetor_alvo = img_alvo.flatten()

print("\n--- PASSO 4: Executando Algoritmo de Difusão (Langevin Anelado) ---")
T = 1000       # Ajustado para 1.000 passos para viabilidade em 200x200
c = 0.005      # Ajustado proporcionalmente para passos maiores devido a queda de T
rng = np.random.default_rng(42)
    
x_processo = rng.standard_normal(vetor_alvo.shape) * sigmas[0]
        
for s in sigmas:
    alpha = c * (s ** 2)
    for _ in range(T):
        z = rng.standard_normal(x_processo.shape)
        score_predito = prever_score_cnn(modelo_cnn, x_processo, s, img_shape)
        x_processo = x_processo + 0.5 * alpha * score_predito + np.sqrt(alpha) * z
                
    # CORREÇÃO CRÍTICA: Reshape dinâmico baseado em img_shape (200, 200, 3) e não fixo em (10,10,3)
img_final = np.clip(x_processo.reshape(img_shape), 0, 1)

print("\n--- PASSO 5: Exibindo e Salvando os Resultados ---")
plt.figure(figsize=(8, 4))
        
plt.subplot(1, 2, 1)
plt.title(f"Entrada {IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}")
plt.imshow(img_alvo)
plt.axis('off')
        
plt.subplot(1, 2, 2)
plt.title("Resultado Difusão")
plt.imshow(img_final)
plt.axis('off')
        
plt.tight_layout()
plt.savefig("resultado_gerado.png")
print("Concluído com Sucesso! Imagem comparativa salva localmente como 'resultado_gerado.png'.")
plt.show()